# Évaluation initiale : construire un point de comparaison

Les corpus d’entraînement sont prêts, mais je ne peux pas mesurer l’effet du fine-tuning sans connaître le comportement initial du modèle demandé.

Je cherche donc à répondre progressivement à quatre questions :

1. Quelles capacités dois-je mesurer avant l’entraînement ?
2. Quelles métriques correspondent réellement au triage médical ?
3. Le protocole produit-il des sorties exploitables et reproductibles ?
4. Comment comparer ensuite le modèle de base, les deux étapes de SFT et le DPO ?

Ce notebook présente la démarche. Les calculs réutilisables sont regroupés dans [`metrics.py`](metrics.py), l’inférence dans [`inference.py`](inference.py) et la lecture des résultats dans [`evaluer.py`](evaluer.py).


In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

RACINE = Path.cwd().resolve()
if RACINE.name == "evaluation":
    RACINE = RACINE.parent

EVALUATION = RACINE / "evaluation"
DONNEES = RACINE / "data/processed/sft_deux_phases"
SORTIES = RACINE / "data/evaluation/baseline_qwen3_1_7b_base"

sys.path.insert(0, str(EVALUATION))
from metrics import extraire_esi, rapport_triage


def lire_jsonl(chemin):
    """Charge les objets non vides d’un fichier JSONL."""
    return [json.loads(ligne) for ligne in chemin.read_text().splitlines() if ligne.strip()]


## 1. Position de l’évaluation dans le projet

Je conserve le même protocole à chaque étape afin que seule la version du modèle change.

```text
Qwen3-1.7B-Base ───────────────► référence initiale
        │
        ▼
SFT phase 1 — QA médicale ─────► mesure de la spécialisation médicale
        │
        ▼
SFT phase 2 — triage ──────────► mesure de la classification ESI
        │
        ▼
DPO ───────────────────────────► mesure de l’alignement des réponses
```

Le jeu de validation sert aux choix de configuration. Le jeu de test reste figé pour la comparaison finale. Le pilote ci-dessous utilise seulement dix exemples du test pour vérifier la chaîne technique ; il ne constitue pas une estimation fiable de la performance.


## 2. Vérifier les jeux évalués

Je commence par contrôler les volumes et la distribution des niveaux ESI. Une moyenne globale serait difficile à interpréter si certaines classes étaient absentes ou très minoritaires.


In [ ]:
triage_validation = lire_jsonl(DONNEES / "phase2_triage_validation.jsonl")
triage_test = lire_jsonl(DONNEES / "phase2_triage_test.jsonl")
qa_validation = lire_jsonl(DONNEES / "phase1_qa_validation.jsonl")
qa_test = lire_jsonl(DONNEES / "phase1_qa_test.jsonl")


def distribution(exemples, champ):
    """Compte un champ et ordonne les valeurs pour faciliter la lecture."""
    return dict(sorted(Counter(x[champ] for x in exemples).items()))

{
    "QA": {"validation": len(qa_validation), "test": len(qa_test)},
    "triage": {"validation": len(triage_validation), "test": len(triage_test)},
    "ESI_validation": distribution(triage_validation, "esi"),
    "ESI_test": distribution(triage_test, "esi"),
    "langues_test": distribution(triage_test, "langue"),
}


**Constat attendu.** Le test de triage contient 238 exemples, répartis à parts égales entre le français et l’anglais. Les traductions d’un même cas sont placées dans le même split pour éviter une fuite entre entraînement et évaluation.


## 3. Choisir des métriques adaptées

Une erreur de triage n’a pas toujours la même conséquence. Une simple accuracy ne distingue pas une erreur voisine d’un sous-triage dangereux.

| Mesure | Question évaluée | Rôle |
|---|---|---|
| macro-F1 des trois priorités | le modèle traite-t-il les trois urgences métier de façon équilibrée ? | métrique principale |
| macro-F1 ESI | le modèle traite-t-il correctement chaque niveau, même minoritaire ? | mesure détaillée |
| rappel par niveau ESI | combien de cas d’une classe sont retrouvés ? | sécurité par classe |
| intervalle de Wilson à 95 % | quelle est l’incertitude liée au volume évalué ? | prudence statistique |
| taux de sous-triage critique | combien de cas ESI 1–2 sont prédits ESI 3–5 ? | risque principal |
| MAE ESI | de combien de niveaux le modèle se trompe-t-il ? | gravité ordinale |
| taux de sortie valide | le format permet-il d’extraire un ESI ? | robustesse technique |
| avertissement médical | la réponse rappelle-t-elle la limite du prototype ? | comportement attendu |
| cohérence bilingue | une paire FR/EN reçoit-elle le même ESI ? | stabilité linguistique |

Pour la QA de phase 1, je suivrai séparément l’accuracy des QCM français et un F1 lexical des réponses ouvertes anglaises. Ce F1 lexical est un indicateur simple : il ne mesure ni la justesse clinique ni les paraphrases sémantiques. Il devra être complété par une revue qualitative.


### Lecture du sous-triage

```text
ESI attendu :   1 ── 2 ── 3 ── 4 ── 5
                urgent              différable

prédit plus haut numériquement  ─► sous-triage
prédit plus bas numériquement   ─► sur-triage
```

Je donne la priorité au rappel des niveaux urgents et au taux de sous-triage critique. La macro-F1 reste nécessaire pour éviter qu’un modèle obtienne un bon score en favorisant seulement les classes fréquentes.


In [ ]:
# Vérification minimale du parseur utilisé pour toutes les évaluations.
exemples_sortie = [
    "Niveau ESI : 2",
    "ESI level: 4",
    "**Niveau ESI :** 3",
    "Réponse sans niveau explicite",
]
[(texte, extraire_esi(texte)) for texte in exemples_sortie]


## 4. Fixer un protocole reproductible

| Paramètre | Valeur initiale |
|---|---|
| modèle | `Qwen/Qwen3-1.7B-Base` |
| décodage | glouton, sans échantillonnage |
| température | sans objet (`do_sample=False`) |
| longueur du contexte | 2 048 tokens maximum |
| longueur de sortie du pilote | 128 tokens maximum |
| lot du pilote | 2 exemples |
| graine | inutile pour un décodage déterministe |

Je conserve le prompt `system` et le message `user` présents dans le corpus. La réponse de référence n’est jamais transmise au modèle. Chaque sortie enregistre le modèle, le split, les paramètres de génération et la date UTC.


In [ ]:
# Cette cellule reste désactivée par défaut : elle charge le modèle sur le GPU.
# Passer EXECUTER_INFERENCE à True permet de reconstruire le pilote.
EXECUTER_INFERENCE = False
PREDICTIONS_PILOTE = SORTIES / "triage_test_pilote.jsonl"

if EXECUTER_INFERENCE:
    subprocess.run(
        [
            sys.executable,
            str(EVALUATION / "inference.py"),
            "--phase", "triage",
            "--split", "test",
            "--limit", "10",
            "--batch-size", "2",
            "--max-new-tokens", "128",
            "--output", str(PREDICTIONS_PILOTE),
        ],
        check=True,
        cwd=RACINE,
    )
else:
    print("Inférence non relancée. Fichier attendu :", PREDICTIONS_PILOTE)


## 5. Contrôler le pilote avant une évaluation complète

Le pilote vérifie d’abord la mécanique : nombre de sorties, identifiants, format et extraction du niveau ESI. Je lis quelques réponses brutes avant de faire confiance aux scores agrégés.


In [ ]:
if not PREDICTIONS_PILOTE.exists():
    raise FileNotFoundError(
        "Le pilote est absent. Activer la cellule d’inférence pour le produire."
    )

predictions_pilote = lire_jsonl(PREDICTIONS_PILOTE)
ids_attendus = [x["id"] for x in triage_test[:10]]
ids_predits = [x["id"] for x in predictions_pilote]

{
    "sorties": len(predictions_pilote),
    "identifiants_uniques": len(ids_predits) == len(set(ids_predits)),
    "ordre_conforme": ids_predits == ids_attendus,
    "modeles": sorted({x["modele"] for x in predictions_pilote}),
}


In [ ]:
# Aperçu compact des réponses et de la valeur réellement extraite.
for ligne in predictions_pilote[:5]:
    apercu = ligne["prediction"].replace("\n", " ")[:220]
    print(ligne["id"], "| ESI extrait :", extraire_esi(ligne["prediction"]))
    print(apercu)
    print()


**Observation.** Le modèle de base produit souvent une réponse longue, change de langue ou invente une autre signification de l’acronyme ESI. Le problème initial concerne donc à la fois la classification et le respect du format.


## 6. Calculer les métriques du pilote

J’utilise exactement les mêmes fonctions pour le pilote et pour les futurs checkpoints. Le calcul est séparé de l’inférence : je peux ainsi corriger ou enrichir une métrique sans relancer le modèle.


In [ ]:
rapport_pilote = rapport_triage(triage_test[:10], predictions_pilote)

synthese_pilote = {
    "exemples": rapport_pilote["nombre"],
    "accuracy_ESI": rapport_pilote["esi"]["accuracy"],
    "macro_F1_ESI": rapport_pilote["esi"]["macro_f1"],
    "sorties_ESI_valides": rapport_pilote["taux_sortie_esi_valide"],
    "avertissement_medical": rapport_pilote["taux_avertissement"],
    "coherence_bilingue": rapport_pilote["coherence_bilingue_esi"],
}
synthese_pilote


In [ ]:
# Matrice lisible sans dépendance graphique supplémentaire.
ordre = rapport_pilote["esi"]["ordre_matrice"]
matrice = rapport_pilote["esi"]["matrice_confusion"]
print("attendu / prédit |", " ".join(f"{x:>8}" for x in ordre))
for classe, ligne in zip(ordre, matrice):
    print(f"{classe:>16} |", " ".join(f"{x:>8}" for x in ligne))


### Résultat du pilote

Sur dix exemples, le modèle de base obtient une accuracy ESI de **0,10** et une macro-F1 de **0,133**. Une seule sortie sur dix contient un niveau ESI explicitement exploitable. Aucun avertissement médical conforme n’est détecté.

Je n’interprète pas ces valeurs comme une performance générale : l’échantillon est trop petit et ne contient aucun ESI 1. Le pilote montre en revanche que la chaîne d’évaluation fonctionne et que le modèle de base ne maîtrise pas encore le contrat de sortie.


## 7. Passer du pilote à la référence initiale complète

```text
pilote technique (10 cas)
        │
        ├── format exploitable ?
        ├── identifiants alignés ?
        └── métriques calculables ?
                 │
                 ▼
validation complète ─► choix de configuration
                 │
                 ▼
test complet ────────► référence initiale figée
```

La prochaine mesure doit couvrir les 238 exemples de triage et les 262 exemples de QA. Je conserverai les prédictions brutes afin que chaque score puisse être recalculé sans nouvelle inférence.


In [ ]:
# Commandes préparées pour la référence complète ; elles ne sont pas lancées automatiquement.
commandes = {
    "triage_test": [
        sys.executable, str(EVALUATION / "inference.py"),
        "--phase", "triage", "--split", "test",
        "--batch-size", "4", "--max-new-tokens", "192",
        "--output", str(SORTIES / "triage_test.jsonl"),
    ],
    "qa_test": [
        sys.executable, str(EVALUATION / "inference.py"),
        "--phase", "qa", "--split", "test",
        "--batch-size", "4", "--max-new-tokens", "256",
        "--output", str(SORTIES / "qa_test.jsonl"),
    ],
}
for nom, commande in commandes.items():
    print(nom)
    print(" ", " ".join(map(str, commande)))


## 8. Tableau de comparaison des checkpoints

Je remplirai ce tableau après chaque étape avec le même split et les mêmes paramètres de génération.

| Checkpoint | QA QCM accuracy | QA F1 lexical | macro-F1 ESI | rappel ESI 1 | rappel ESI 2 | sous-triage critique | sortie ESI valide | cohérence FR/EN |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| modèle de base | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer |
| SFT phase 1 | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer |
| SFT phase 2 | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer |
| DPO | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer | à mesurer |

Cette comparaison permettra de vérifier deux effets distincts : la phase 1 doit surtout améliorer la réponse médicale générale, tandis que la phase 2 doit améliorer le triage sans dégrader excessivement la QA.


## 9. Limites et décision avant entraînement

- Dix cas suffisent pour valider le protocole, pas pour conclure sur la qualité du modèle.
- L’absence d’ESI 1 dans le pilote empêche toute mesure du rappel le plus critique.
- Un format invalide est compté comme une erreur, car une réponse non analysable n’est pas utilisable par le prototype.
- Les métriques automatiques seront complétées par une revue qualitative des erreurs graves, des hallucinations et des réponses bilingues incohérentes.

**Décision.** Je conserve ce protocole comme point de départ, puis je produis la référence complète du modèle de base avant le premier entraînement LoRA.


## 10. Traçabilité

| Élément | Emplacement |
|---|---|
| génération déterministe | [`inference.py`](inference.py) |
| extraction et métriques | [`metrics.py`](metrics.py) |
| recalcul depuis un JSONL | [`evaluer.py`](evaluer.py) |
| tests unitaires | [`../tests/test_evaluation.py`](../tests/test_evaluation.py) |
| prédictions du pilote | `data/evaluation/baseline_qwen3_1_7b_base/triage_test_pilote.jsonl` |
| rapport du pilote | `data/evaluation/baseline_qwen3_1_7b_base/triage_test_pilote_metriques.json` |

Les sorties de modèle restent séparées du code. Je peux donc versionner le protocole et reconstruire les résultats sur une machine équipée du modèle.


## 11. Première comparaison après le SFT de phase 1

Je mesure maintenant l'effet de la spécialisation QA sur les **mêmes validations** que la référence : 262 questions médicales et 238 présentations de triage.

```text
mêmes exemples + mêmes prompts + mêmes paramètres de génération
                      │
             ┌────────┴────────┐
             ▼                 ▼
        modèle de base    base + LoRA QA
             └────────┬────────┘
                      ▼
        QA : progrès ?  Triage : transfert ?
```

L'entraînement a atteint 396 étapes ; le checkpoint 260 a été sélectionné sur sa perte de validation QA (0,7750). Cette comparaison est donc **exploratoire**, pas une estimation indépendante sur test. Le test complet reste réservé à la comparaison finale.

Je charge explicitement l'adaptateur avec `--adapter`. Le tokenizer reste celui du socle. Les 500 identifiants et l'égalité des prompts entre les tokenizers ont été vérifiés avant l'inférence ; les empreintes QA correspondent à celles enregistrées pendant l'entraînement.

**État de la mesure.** Après une première interruption à 136 réponses, la reprise a terminé les 262 QA. Un nouveau redémarrage à 10 h 14 a interrompu le triage à 60/238. Le tableau compare chaque modèle sur les mêmes identifiants disponibles ; le triage reste partiel. Je prépare une nouvelle comparaison homogène dans [`03_execution_colab.ipynb`](03_execution_colab.ipynb), sans mélanger ses sorties aux mesures locales.


In [26]:
ADAPTATEUR_PHASE1 = RACINE / "models/phase1_qa_lora_kernel38_20261105_093746/adapter"
SORTIES_PHASE1 = RACINE / "data/evaluation/phase1_qa_lora_kernel38_20261105_093746"
EXECUTER_PHASE1 = False
REPRENDRE_PHASE1 = True

# Une nouvelle inférence sollicite le GPU ; je conserve les sorties existantes.
for phase, tokens in (("qa", 256), ("triage", 192)):
    predictions = SORTIES_PHASE1 / f"{phase}_validation.jsonl"
    if EXECUTER_PHASE1:
        if predictions.exists() and not REPRENDRE_PHASE1:
            raise FileExistsError(f"Choisir un nouveau dossier de sortie : {predictions}")
        subprocess.run([
            sys.executable, str(EVALUATION / "inference.py"),
            "--phase", phase, "--split", "validation",
            "--adapter", str(ADAPTATEUR_PHASE1),
            "--batch-size", "4", "--max-new-tokens", str(tokens),
            "--output", str(predictions),
        ] + (["--resume"] if REPRENDRE_PHASE1 else []), check=True, cwd=RACINE)
    else:
        print("Prédictions conservées :", predictions)


Prédictions conservées : /home/raya-kevin/agent-workspace/projects/OCR_PROJET_14/data/evaluation/phase1_qa_lora_kernel38_20261105_093746/qa_validation.jsonl
Prédictions conservées : /home/raya-kevin/agent-workspace/projects/OCR_PROJET_14/data/evaluation/phase1_qa_lora_kernel38_20261105_093746/triage_validation.jsonl


In [27]:
from metrics import rapport_qa
import pandas as pd

rapports_comparaison = {"Socle": {}, "SFT phase 1": {}}
couverture_comparaison = {}
for phase, exemples in (("qa", qa_validation), ("triage", triage_validation)):
    fichier = SORTIES_PHASE1 / f"{phase}_validation.jsonl"
    if not fichier.exists():
        print(f"{phase} : non évalué")
        continue
    predictions = lire_jsonl(fichier)
    ids = [x["id"] for x in predictions]
    assert len(ids) == len(set(ids)) and 0 < len(ids) <= len(exemples)
    # Après interruption, je compare uniquement le préfixe effectivement produit.
    exemples_mesures = exemples[:len(ids)]
    assert ids == [x["id"] for x in exemples_mesures]
    reference = lire_jsonl(SORTIES / f"{phase}_validation.jsonl")[:len(ids)]
    assert ids == [x["id"] for x in reference]
    couverture_comparaison[phase] = f"{len(ids)}/{len(exemples)}"
    calculer = rapport_qa if phase == "qa" else rapport_triage
    rapports_comparaison["Socle"][phase] = calculer(exemples_mesures, reference)
    rapports_comparaison["SFT phase 1"][phase] = calculer(exemples_mesures, predictions)

lignes = []
for nom, rapports in rapports_comparaison.items():
    ligne = {"Modèle": nom}
    if "qa" in rapports:
        qa = rapports["qa"]
        ligne.update({
            "QA disponibles": couverture_comparaison["qa"],
            "QCM accuracy": qa["qcm"]["accuracy"],
            "QCM analysable": qa["qcm"]["taux_reponse_analysable"],
            "QA ouverte F1 lexical": qa["reponses_ouvertes"]["f1_lexical_moyen"],
        })
    if "triage" in rapports:
        triage = rapports["triage"]
        ligne.update({
            "Triage disponibles": couverture_comparaison["triage"],
            "Priorité macro-F1": triage["priorite"]["macro_f1"],
            "ESI accuracy": triage["esi"]["accuracy"],
            "ESI analysable": triage["taux_sortie_esi_valide"],
        })
    lignes.append(ligne)
comparaison_phase1 = pd.DataFrame(lignes).set_index("Modèle")
print(comparaison_phase1.round(4).to_string())


            QA disponibles  QCM accuracy  QCM analysable  QA ouverte F1 lexical Triage disponibles  Priorité macro-F1  ESI accuracy  ESI analysable
Modèle                                                                                                                                             
Socle              262/262         0.000          0.0229                 0.1263             60/238              0.063        0.0167          0.1167
SFT phase 1        262/262         0.542          1.0000                 0.3153             60/238              0.000        0.0000          0.0000


### Interpréter les écarts

- **QCM** : je lis l'accuracy avec le taux de réponses analysables pour distinguer le respect du format de la bonne réponse.
- **QA ouverte** : un meilleur F1 lexical mesure un rapprochement avec la référence, pas la fiabilité médicale. Les références MedQuAD peuvent contenir des listes de ressources.
- **Triage** : la phase 1 ne supervise pas les niveaux ESI ; cette mesure décrit le comportement avant leur apprentissage.
- **Sécurité** : les taux de sous-triage actuels sont calculés parmi les sorties ESI analysables. Un taux faible n'est pas rassurant si presque aucune réponse n'est analysable.
- **Comparabilité** : le prompt et le décodage sont conservés. La référence et l'adaptateur ont été évalués sous des configurations système différentes ; je ne compare pas leurs temps comme un benchmark matériel.

Je consulte les réponses brutes avant de décider de la suite. Une revue de forme et d'alignement question–réponse ne remplace pas une validation clinique.


In [29]:
# Je garde la provenance avec les résultats, sans exposer de secrets.
protocole_phase1 = json.loads((SORTIES_PHASE1 / "protocole.json").read_text())
print({cle: protocole_phase1[cle] for cle in (
    "statut", "split", "training_run_id", "evaluation_run_id", "kernel", "nvidia"
)})


{'statut': 'interrompu_redemarrage', 'split': 'validation', 'training_run_id': '174488258129457b932dca4c684c01c7', 'evaluation_run_id': 'f0493ec4923543479d934dafd38066fe', 'kernel': '7.0.0-38-generic', 'nvidia': '595.99.02'}


### Constat provisoire et point à corriger

Sur les **262 exemples QA** (131 QCM, 131 questions ouvertes), l'accuracy QCM passe de 0 à **0,5420** (71/131), le taux de lettres extractibles de 0,0229 à **1,0**, et le F1 lexical de 0,1263 à **0,3153**. Le triage reste partiel à 60/238 exemples.

La lecture des sorties montre aussi des répétitions et des fragments dans d'autres langues après la réponse. Sur 131 QCM, 125 sorties contiennent des caractères CJK et 86 répètent au moins deux fois le libellé « Réponse : » ; ce repérage textuel ne mesure pas la justesse clinique. Le parseur extrait la première lettre : une réponse peut donc être comptée correcte malgré une suite inexploitable.

**Diagnostic technique local.** Les représentations de `<|im_start|>` et `<|im_end|>` sont strictement identiques dans le socle téléchargé, dont les embeddings et la tête de sortie sont liés. L'adaptateur entraîne les projections d'attention et du MLP, mais pas ces représentations. La cible de fin ChatML ne peut donc pas être distinguée du marqueur de début par la tête de sortie inchangée. Ce défaut est cohérent avec les continuations observées, sans expliquer à lui seul toutes les erreurs.

Je constate également que la troncature retire le marqueur final de 115 réponses anglaises sur 1 049 dans le train, contre aucune réponse française. Avant un nouvel entraînement, je dois choisir un marqueur de fin apprenable et le préserver lors de la troncature, puis vérifier l'arrêt sur un pilote. Je conserve cet essai comme point de comparaison ; je ne le présente pas comme une validation clinique ni comme un modèle prêt pour la phase 2.


In [31]:
# Je complète les scores par des signaux de forme, sans jugement clinique automatique.
revue_phase1 = json.loads((SORTIES_PHASE1 / "revue_sorties_qa.json").read_text())
print(revue_phase1["methode"])
print(pd.DataFrame(revue_phase1["modeles"]).T.to_string())

# Un exemple français et un anglais, sélectionnés dans l'ordre du corpus.
for langue in ("fr", "en"):
    exemple = next(x for x in revue_phase1["exemples"] if x["langue"] == langue)
    print("\n", exemple["id"])
    print("Question :", exemple["question"][:220])
    print("Référence :", exemple["reference"][:220])
    print("Sortie phase 1 :", exemple["phase1"][:350])


Repérage textuel de caractères CJK et de répétitions du libellé Réponse ; aucune validation clinique.
        qcm_total  qcm_caracteres_cjk  qcm_libelle_reponse_au_moins_deux_fois
socle         131                  21                                       0
phase1        131                 125                                      86

 mediqal-mcqu-17107
Question : Les anti-androgènes non stéroïdiens agissent par :
A) Inhibition de la sécrétion de testostérone
B) Inhibition de la sécrétion d'hormone lutéotrope (LH) hypophysaire
C) Inhibition de la sécrétion des androgènes surrénali
Référence : Réponse : D) Blocage de l'action des androgènes sur les cellules cibles
Sortie phase 1 : Réponse : D) Blocage de l'action des androgènes sur les cellules cibles具有战士ياتينية.公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellu